# Feature Engineering : Préparation pour le Machine Learning

Ce notebook prend le fichier  et le transforme en , prêt à être ingéré par des modèles prédictifs (XGBoost, Random Forest, Régression Linéaire).

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Configuration
pd.set_option("display.max_columns", None)

print("Libraries chargées avec succès.")

## 1. Chargement des données propres

In [ ]:
df = pd.read_csv("../data/clean_data.csv")
print(f"Dimensions: {df.shape[0]} lignes, {df.shape[1]} colonnes.")
df.head(3)

## 2. Sélection des Variables (Feature Selection)
On retire les identifiants textuels (qui ne servent à rien pour l'algorithme) et les variables temporelles brutes.

In [ ]:
cols_to_drop = [
    "id", "title", "description", "publishedAt", 
    "channelId", "channel_id", "channel_name", 
    "date", "thumbnails", "tags", "topics", "duration", "avg_like_rate", "avg_duration_sec", "pct_vertical", "pct_has_music", "viral_video_rate", "consistency_score"
]

# On garde df_features pour l'entraînement pur
df_features = df.drop(columns=cols_to_drop, errors="ignore")

print(f"Colonnes conservées pour l'apprentissage : {df_features.shape[1]}")
list(df_features.columns)

## 3. Encodage des Variables Catégoriques
L'algorithme ne comprend que les chiffres. Si nous avons des catégories (comme le ), il faut les transformer en 0, 1, 2, etc.

In [ ]:
# Vérifions s'il y a des colonnes non numériques
categorical_cols = df_features.select_dtypes(include=["object"]).columns
print("Colonnes nécessitant un encodage :", list(categorical_cols))

# Si creator_tier est présent, on peut faire un One-Hot Encoding ou un Label Encoding
if "creator_tier" in categorical_cols:
    # Get Dummies convertit "Bronze", "Gold" en colonnes tier_Bronze (0/1), tier_Gold (0/1)
    df_features = pd.get_dummies(df_features, columns=["creator_tier"], drop_first=True)
    print("creator_tier encodé avec succès.")

df_features.head()

## 4. Nettoyage des fuites de données (Data Leakage)
**ATTENTION CRITIQUE** : La variable cible  a été calculée mathématiquement à partir de , , et .
Si on laisse  et  dans les variables d'entraînement, l'algorithme va "tricher" à 100% en devinant la formule ! Seule compte la capacité à prédire la viralité A PARTIR des attributs de la vidéo (titre long, émoticônes, âge de la chaîne...), et non à partir de ses performances futures.

In [ ]:
leakage_cols = ["views", "likes", "comments", "views_per_subscriber"]
df_ml = df_features.drop(columns=[col for col in leakage_cols if col in df_features.columns])

print(f"Attributs prédictifs finaux ({df_ml.shape[1]} colonnes) :")
print(list(df_ml.columns))

## 5. Corrélation avec la Target
Regardons quelles sont les variables qui influencent le plus le score de viralité !

In [ ]:
plt.figure(figsize=(16, 12))
# Calcul de la matrice de corrélation complète
corr_matrix = df_ml.corr()

# Création d'un masque pour cacher la moitié supérieure symétrique (optionnel mais plus lisible)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Heatmap seaborn de la matrice
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", 
            vmax=1, vmin=-1, center=0, square=True, linewidths=.5, cbar_kws={"shrink": .5})

plt.title("Matrice de Corrélation Complète des Features", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()


## 6. Sauvegarde des Features
Les données sont prêtes à être envoyées à Scikit-Learn ou XGBoost.

In [ ]:
df_ml.to_csv("../data/features.csv", index=False)
print("Fichier features.csv sauvegardé avec succès dans le dossier data ! ")